# Experiment 4: Few-Shot Examples

## Research Question
How many few-shot examples improve intertextuality detection performance?

### Fixed Parameters
- Model: gpt-5.4-nano
- Prompt: zero_shot_maximum -> best prompt from experiment 1-> (minimal for gpt-4o-mini)
- Chunk size: 120 -> best from experiment 2 ->120 for gpt-4o-mini
- Overlap: 20 ->best from experiment 3 in gpt-5.4-nano -> (20 for gpt-4o-mini)

### Variable Parameter
- num_few_shot_examples ∈ {8}

### used Models for analysis
- (gpt-5.4-mini)
- (gpt-5.4)
- gpt-5.4-nano
- (gpt-4o-mini)

In [ ]:
import sys
project_src = r"C:\Users\sophi\Desktop\Master\CIS_5.Semester\Thesis\locisimiles\src"

# lokalen Pfad *vor* site-packages setzen
sys.path.insert(0, project_src)

import locisimiles
print("Jetzt geladen aus:", locisimiles.__file__)

#imports
import openai
from unittest.mock import patch
import locisimiles as ls
from locisimiles.evaluator import IntertextEvaluator
#from locisimiles.llm_pipeline_new import LLMOnlyClassificationPipeline
from locisimiles.llm_pipeline_sliding_windows import LLMOnlySlidingWindowPipeline, pretty_print, save_final_results, load_final_results
from locisimiles.document import Document, TextSegment
#from locisimiles.llm_pipeline_new import pretty_print
from locisimiles.llm_pipeline_sliding_windows import convert_llm_results, fixed_sliding_windows, convert_llm_results_coverage_fair, pad_segments
import pandas as pd
import csv
from pathlib import Path
from locisimiles.evaluator import IntertextEvaluator
import random
import tempfile
import time

# load key
from dotenv import load_dotenv
load_dotenv()

: 

In [2]:
# =========================
# DATA PATH
# =========================
DATA_ROOT = Path("../Sampled_Documents/folds")

N_FOLDS = 5

In [3]:
def load_fold(
    fold_id: int,
    *,
    root: Path = DATA_ROOT,
):
    """
    Load query document, source document and ground truth for one fold.
    """

    fold_dir = root / f"fold_{fold_id}" / "evaluation"

    if not fold_dir.exists():
        raise FileNotFoundError(f"Fold {fold_id} not found at {fold_dir}")

    query_doc = Document(fold_dir / "query_document.csv")
    source_doc = Document(fold_dir / "source_document.csv")
    ground_truth_path = fold_dir / "ground_truth.csv"

    return {
        "query": query_doc,
        "source": source_doc,
        "ground_truth": ground_truth_path,
    }

In [ ]:
# =========================
# FIXED PARAMETERS
# =========================

MODEL = "gpt-5.4-nano"
BEST_PROMPT = "zero_shot_maximum" #replace with result from Experiment 1->best result
BEST_CHUNK = 120 #replace with result from Experiment 2-> best chunk
BEST_OVERLAP = 20 #replace with result from Experiment 3 -> best overlap

# =========================
# VARIABLE/EXPERIMENT PARAMETER
# =========================
FEW_SHOT_COUNTS = [8]

# ==============================
# OUTPUT
# ==============================
# change result path for each model
#RESULT_DIR = Path("final_results/experiment_4_few_shot_gpt-5.4-mini")
#RESULT_DIR = Path("final_results/experiment_4_few_shot_gpt-5.4")
RESULT_DIR = Path("final_results/experiment_4_few_shot_gpt-5.4-nano")
#RESULT_DIR = Path("final_results/experiment_4_few_shot_gpt-4o-mini")
RESULT_DIR.mkdir(parents=True, exist_ok=True)

In [5]:
# =========================
# RUN EXPERIMENT
# =========================
for k in FEW_SHOT_COUNTS:

    print("\n====================================")
    print(f"Running experiment for few_shot: {k}")
    print("====================================")

    pipeline = LLMOnlySlidingWindowPipeline(
        openai_model=MODEL,
        prompt_type=BEST_PROMPT,
        chunk_size=BEST_CHUNK,
        overlap=BEST_OVERLAP,
        num_few_shot_examples=k,
    )

    for fold_id in range(N_FOLDS):

        print(f"\nRunning fold {fold_id}")

        fold = load_fold(fold_id)

        checkpoint_path = RESULT_DIR / f"checkpoint_fold_{fold_id}_fewShot{k}.json"
        
        start_time = time.perf_counter()

        results = pipeline.run(
            query=fold["query"],
            source=fold["source"],
            checkpoint_path=checkpoint_path,
            resume=True,
        )
        
        end_time = time.perf_counter()
        runtime = end_time - start_time
        print(f"Fold {fold_id} runtime: {runtime:.2f} seconds")

        out_path = RESULT_DIR / f"fold_{fold_id}_few_shot_{k}.json"

        save_final_results(
            results,
            path=out_path,
            meta={
                "model": pipeline.openai_model,
                "prompt_type": BEST_PROMPT,
                "chunk_size": pipeline.chunk_size,
                "num_few_shot_examples": pipeline.num_few_shot_examples,
                "overlap": pipeline.overlap,
                "runtime_seconds": runtime,
            },
        )

        print(f"Saved results to {out_path}")


Running experiment for few_shot: 8

Running fold 0
Total source windows: 10


Processing source windows:   0%|          | 0/10 [00:00<?, ?it/s]

Processing source windows: 100%|██████████| 10/10 [19:20<00:00, 116.06s/it]


Fold 0 runtime: 1160.54 seconds
Saved results to final_results\experiment_4_few_shot_gpt-5.4-mini\fold_0_few_shot_8.json

Running fold 1
Total source windows: 10


Processing source windows: 100%|██████████| 10/10 [16:30<00:00, 99.08s/it] 


Fold 1 runtime: 990.76 seconds
Saved results to final_results\experiment_4_few_shot_gpt-5.4-mini\fold_1_few_shot_8.json

Running fold 2
Total source windows: 10


... (2 outputs left out)

Processing source windows: 100%|██████████| 10/10 [17:54<00:00, 107.48s/it]


Fold 3 runtime: 1074.82 seconds
Saved results to final_results\experiment_4_few_shot_gpt-5.4-mini\fold_3_few_shot_8.json

Running fold 4
Total source windows: 10


Processing source windows: 100%|██████████| 10/10 [19:22<00:00, 116.25s/it]


Fold 4 runtime: 1162.55 seconds
Saved results to final_results\experiment_4_few_shot_gpt-5.4-mini\fold_4_few_shot_8.json


In [6]:
# =========================
# EVALUATION
# =========================
all_scores = []

for k in FEW_SHOT_COUNTS:

    print("\n===============================")
    print(f"Evaluating few shot: {k}")
    print("===============================")

    for fold_id in range(N_FOLDS):

        print(f"\nEvaluating fold {fold_id}")

        fold = load_fold(fold_id)

        result_file = RESULT_DIR / f"fold_{fold_id}_few_shot_{k}.json"

        raw_results = load_final_results(result_file)
        
        pipeline = LLMOnlySlidingWindowPipeline(
            openai_model=MODEL,
            prompt_type=BEST_PROMPT,
            chunk_size=BEST_CHUNK,
            overlap=BEST_OVERLAP,
            num_few_shot_examples=k,
        )


        source_segments = pad_segments(
            list(fold["source"].segments.values()),
            BEST_CHUNK,
            prefix="S"
        )

        query_segments = pad_segments(
            list(fold["query"].segments.values()),
            BEST_CHUNK,
            prefix="Q"
        )

        source_windows = fixed_sliding_windows(
            source_segments,
            BEST_CHUNK,
            pipeline.stride
        )

        query_windows = fixed_sliding_windows(
            query_segments,
            BEST_CHUNK,
            pipeline.stride
        )

        full_results = convert_llm_results_coverage_fair(
            raw_results,
            fold["source"],
            fold["query"],
            source_windows,
            query_windows,
            threshold=0.0
        )

        wrapped_pipeline = type(
            "DummyPipeline",
            (),
            {"run": lambda self, **kwargs: full_results}
        )()

        evaluator = IntertextEvaluator(
            query_doc=fold["query"],
            source_doc=fold["source"],
            ground_truth_csv=fold["ground_truth"],
            pipeline=wrapped_pipeline,
            top_k=1000,
            threshold=0.5,
        )

        macro_scores = evaluator.evaluate(average="macro")
        micro_scores = evaluator.evaluate(average="micro")

        print("Macro:", macro_scores)
        print("Micro:", micro_scores)
        
        macro_row = macro_scores.iloc[0]
        micro_row = micro_scores.iloc[0]

        all_scores.append({
            "few_shot": k,
            "fold": fold_id,
            "macro_precision": macro_row["precision"],
            "macro_recall": macro_row["recall"],
            "macro_f1": macro_row["f1"],
            "macro_accuracy": macro_row["accuracy"],
            "macro_fpr": macro_row["fpr"],
            "macro_fnr": macro_row["fnr"],
            "macro_smr": macro_row["smr"],
            "macro_tp": macro_row["tp"],
            "macro_fp": macro_row["fp"],
            "macro_fn": macro_row["fn"],
            "macro_tn": macro_row["tn"],

            "micro_precision": micro_row["precision"],
            "micro_recall": micro_row["recall"],
            "micro_f1": micro_row["f1"],
            "micro_accuracy": micro_row["accuracy"],
            "micro_fpr": micro_row["fpr"],
            "micro_fnr": micro_row["fnr"],
            "micro_smr": micro_row["smr"],
            "micro_tp": micro_row["tp"],
            "micro_fp": micro_row["fp"],
            "micro_fn": micro_row["fn"],
            "micro_tn": micro_row["tn"],
        })


Evaluating few shot: 8

Evaluating fold 0
Macro:    precision    recall        f1  accuracy       fpr       fnr       smr  tp  \
0   0.032373  0.037353  0.033974  0.999894  0.000017  0.000089  0.000106  36   

   fp  fn      tn  
0  14  73  824437  
Micro:    precision    recall       f1  accuracy       fpr       fnr       smr  tp  \
0       0.72  0.330275  0.45283  0.999894  0.000017  0.000089  0.000106  36   

   fp  fn      tn  
0  14  73  824437  

Evaluating fold 1
Macro:    precision   recall        f1  accuracy       fpr       fnr      smr  tp  \
0    0.02754  0.03262  0.028948   0.99988  0.000026  0.000095  0.00012  31   

   fp  fn      tn  
0  21  78  822670  
Micro:    precision    recall        f1  accuracy       fpr       fnr      smr  tp  \
0   0.596154  0.284404  0.385093   0.99988  0.000026  0.000095  0.00012  31   

   fp  fn      tn  
0  21  78  822670  

Evaluating fold 2
Macro:    precision    recall        f1  accuracy       fpr       fnr       smr  tp  \
0   0.04

In [6]:
result_file = RESULT_DIR / f"fold_{fold_id}_few_shot_{k}.json"
result_file

WindowsPath('results/experiment_4_few_shot/fold_0_few_shot_8.json')

In [7]:
# =========================
# RESULTS AGGREGATION
# =========================
df = pd.DataFrame(all_scores)

summary = (
    df.groupby("few_shot")
    .mean(numeric_only=True)
    .sort_values("macro_f1", ascending=False)
)

summary

,fold,macro_precision,macro_recall,macro_f1,macro_accuracy,macro_fpr,macro_fnr,macro_smr,macro_tp,macro_fp,...,micro_recall,micro_f1,micro_accuracy,micro_fpr,micro_fnr,micro_smr,micro_tp,micro_fp,micro_fn,micro_tn
few_shot,,,,,,,,,,,,,,,,,,,,,
8,2.0,0.03575,0.038917,0.036704,0.999897,0.000015,0.000087,0.000103,37.0,12.6,...,0.340061,0.467677,0.999897,0.000015,0.000087,0.000103,37.0,12.6,71.8,823030.6


In [8]:
# =========================
# SAVE RESULT EVALUATION
# =========================
summary_path = RESULT_DIR / "few_shot_experiment_summary.csv"

summary.to_csv(summary_path)
df.to_csv(RESULT_DIR / "few_shot_experiment_results.csv", index=False)

print("Saved summary to", summary_path)

Saved summary to final_results\experiment_4_few_shot_gpt-5.4-mini\few_shot_experiment_summary.csv
